# PCA in Practice with MNIST

## Focus
In this activity, you will explore Principal Component Analysis (PCA) on the MNIST dataset.

Your goals are to:

Understand how PCA works in practice

Explore the impact of dimensionality reduction on training time and model performance

Reflect on when PCA is effective in machine learning workflows

## Dataset
MNIST dataset of handwritten digits

Training set: first 60,000 images

Test set: remaining 10,000 images

## Part 1 — Implementation

### Step 1 — Baseline Random Forest

1. Load the MNIST dataset and split into training and test sets.

In [2]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist["data"], mnist["target"].astype(int)

X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

In [3]:
# applying standardScaleer
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
#X_train = scaler.fit_transform(X_train)
#X_test = scaler.transform(X_test)

2. Train a Random Forest classifier on the full dataset.

In [4]:
from sklearn.ensemble import RandomForestClassifier
import time
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
start = time.perf_counter()
rf_clf.fit(X_train, y_train)
rf_clf_training_time = time.perf_counter() - start

3. Record the training time.

In [5]:
print("Random Forest Classifier training time:", round(rf_clf_training_time, 1))

Random Forest Classifier training time: 3.5


4. Evaluate the classifier on the test set (accuracy, confusion matrix, or other relevant metrics)

In [9]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = rf_clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification report:")
print(classification_report(y_test, y_pred))

Test accuracy: 0.9705
Confusion matrix:
[[ 971    0    0    0    0    2    3    1    3    0]
 [   0 1127    2    2    0    1    2    0    1    0]
 [   6    0 1002    5    3    0    3    8    5    0]
 [   1    0    9  972    0    9    0    9    8    2]
 [   1    0    0    0  955    0    5    1    4   16]
 [   5    1    1    9    2  860    5    2    5    2]
 [   7    3    0    0    3    3  937    0    5    0]
 [   1    4   20    2    0    0    0  990    2    9]
 [   4    0    6    7    5    5    5    4  930    8]
 [   7    6    2   12   12    1    0    4    4  961]]
Classification report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       980
           1       0.99      0.99      0.99      1135
           2       0.96      0.97      0.97      1032
           3       0.96      0.96      0.96      1010
           4       0.97      0.97      0.97       982
           5       0.98      0.96      0.97       892
           6       0.98    

### Step 2 — Apply PCA

1. Use PCA to reduce the dataset’s dimensionality, keeping 95% explained variance

In [6]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("Original n_features:", X_train.shape[1], "Reduced:", X_train_pca.shape[1])

Original n_features: 784 Reduced: 154


2. Train a new Random Forest classifier on the reduced dataset.

In [7]:
rf_clf_pca = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
start = time.perf_counter()
rf_clf_pca.fit(X_train_pca, y_train)
rf_clf_pca_training_time = time.perf_counter() - start

3. Record the training time

In [8]:
print("Random Forest Classifier with PCA training time:", round(rf_clf_pca_training_time, 1))

Random Forest Classifier with PCA training time: 9.8


4. Evaluate the classifier on the test set.

In [9]:
y_pred = rf_clf_pca.predict(X_test_pca)

acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification report:")
print(classification_report(y_test, y_pred))

Test accuracy: 0.9481
Confusion matrix:
[[ 965    0    2    0    0    3    7    1    2    0]
 [   0 1117    4    5    0    1    3    0    4    1]
 [  10    0  967   10    6    1    4    8   23    3]
 [   1    0   11  952    1   14    2    9   14    6]
 [   1    1    5    0  934    3    9    0    5   24]
 [   4    1    3   26    4  832   10    3    4    5]
 [   6    3    2    0    4    9  933    0    1    0]
 [   1    7   17    4    5    0    0  969    1   24]
 [   6    0    8   21   10   19    8    5  886   11]
 [   6    6    3   16   29    5    1   11    6  926]]
Classification report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       980
           1       0.98      0.98      0.98      1135
           2       0.95      0.94      0.94      1032
           3       0.92      0.94      0.93      1010
           4       0.94      0.95      0.95       982
           5       0.94      0.93      0.94       892
           6       0.95    

### Questions to Consider:
- Was training significantly faster?

No, for Random Forestm, training took longer with PCA, from 3.7s with baseline to 10.2 with PCA 

- How did the model’s performance compare to the baseline?

With the performance, with model's test accuracy dropped with PCA, from 97.05% with baseline to 94.81% with PCA

### Step 3 — Try with SGDClassifier

1. Train an SGDClassifier on the full dataset and record training time and performance.

In [10]:
from sklearn.linear_model import SGDClassifier

sgd_clf = SGDClassifier(random_state=42, max_iter=1000, tol=1e-3)

start = time.perf_counter()
sgd_clf.fit(X_train, y_train)
sgd_training_time = time.perf_counter() - start

y_pred = sgd_clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")
print(f"Training time: {sgd_training_time:.2f} seconds")

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification report:")
print(classification_report(y_test, y_pred))

Test accuracy: 0.8740
Training time: 100.76 seconds
Confusion matrix:
[[ 902    0    8   11    1   13    2    4   39    0]
 [   0 1095    2    3    0    2    4    1   28    0]
 [   1   10  803   69    6    4    4   10  122    3]
 [   0    1    6  931    1   21    3    7   35    5]
 [   2    2    9   15  778    4    2    9   62   99]
 [   6    2    1   71    3  709   12   12   67    9]
 [   5    3   12   13    5   21  854    0   45    0]
 [   0    3   18   20    3    4    1  919   18   42]
 [   3    5    2   30    4   43    5    5  872    5]
 [   3    5    2   33    7    5    0   20   57  877]]
Classification report:
              precision    recall  f1-score   support

           0       0.98      0.92      0.95       980
           1       0.97      0.96      0.97      1135
           2       0.93      0.78      0.85      1032
           3       0.78      0.92      0.84      1010
           4       0.96      0.79      0.87       982
           5       0.86      0.79      0.83       8

2. Train the same classifier on the PCA-reduced dataset.

In [11]:
sgd_clf_pca = SGDClassifier(random_state=42, max_iter=1000, tol=1e-3)

start = time.perf_counter()
sgd_clf_pca.fit(X_train_pca, y_train)
sgd_pca_training_time = time.perf_counter() - start

y_pred_pca = sgd_clf_pca.predict(X_test_pca)

acc_pca = accuracy_score(y_test, y_pred_pca)
print(f"Test accuracy (PCA): {acc_pca:.4f}")
print(f"Training time (PCA): {sgd_pca_training_time:.2f} seconds")

print("Confusion matrix (PCA):")
print(confusion_matrix(y_test, y_pred_pca))

print("Classification report (PCA):")
print(classification_report(y_test, y_pred_pca))

Test accuracy (PCA): 0.8959
Training time (PCA): 24.08 seconds
Confusion matrix (PCA):
[[ 933    0    9    1    4    2   18    8    2    3]
 [   0 1099    6    4    1    4    3    1   17    0]
 [   7    6  917   10   14    3   19   11   40    5]
 [   6    5   33  893    2   21    5   13   23    9]
 [   1    3    4    3  901    2   12    7    7   42]
 [  20    4    6   37   16  719   22   11   44   13]
 [   9    2    7    1    5   21  908    3    2    0]
 [   0   11   17   11    8    2    1  928    4   46]
 [  12   11   15   17    8   30   16   12  822   31]
 [   7   11    5   10   44   11    0   62   20  839]]
Classification report (PCA):
              precision    recall  f1-score   support

           0       0.94      0.95      0.94       980
           1       0.95      0.97      0.96      1135
           2       0.90      0.89      0.89      1032
           3       0.90      0.88      0.89      1010
           4       0.90      0.92      0.91       982
           5       0.88     

3. Compare results.

In [12]:
rf_acc = accuracy_score(y_test, rf_clf.predict(X_test))
rf_pca_acc = accuracy_score(y_test, rf_clf_pca.predict(X_test_pca))

sgd_acc = accuracy_score(y_test, sgd_clf.predict(X_test))
sgd_pca_acc = accuracy_score(y_test, sgd_clf_pca.predict(X_test_pca))

print("Random Forest (full)\t- acc: {:.4f}, train time: {:.2f}s".format(rf_acc, rf_clf_training_time))
print("Random Forest (PCA)\t- acc: {:.4f}, train time: {:.2f}s".format(rf_pca_acc, rf_clf_pca_training_time))
print()
print("SGDClassifier (full)\t- acc: {:.4f}, train time: {:.2f}s".format(sgd_acc, sgd_training_time))
print("SGDClassifier (PCA)\t- acc: {:.4f}, train time: {:.2f}s".format(sgd_pca_acc, sgd_pca_training_time))

Random Forest (full)	- acc: 0.9705, train time: 3.39s
Random Forest (PCA)	- acc: 0.9481, train time: 9.79s

SGDClassifier (full)	- acc: 0.8740, train time: 100.76s
SGDClassifier (PCA)	- acc: 0.8959, train time: 24.08s


old results without StandardScaler:

Random Forest (full)	- acc: 0.9705, train time: 3.68s
Random Forest (PCA)	- acc: 0.9481, train time: 10.24s

SGDClassifier (full)	- acc: 0.8740, train time: 124.15s
SGDClassifier (PCA)	- acc: 0.8959, train time: 24.36s

In [10]:
# training XGBoost
from xgboost import XGBClassifier

xgb_clf = XGBClassifier(
    objective='multi:softmax',
    num_class=10,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)

start = time.perf_counter()
xgb_clf.fit(X_train, y_train)
xgb_training_time = time.perf_counter() - start

y_pred = xgb_clf.predict(X_test)
xgb_acc = accuracy_score(y_test, y_pred)
print(f"XGBoost accuracy: {xgb_acc:.4f}")
print(f"XGBoost training time: {xgb_training_time:.2f} seconds")

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification report:")
print(classification_report(y_test, y_pred))


XGBoost accuracy: 0.9717
XGBoost training time: 60.53 seconds
Confusion matrix:
[[ 970    1    0    0    0    2    2    1    3    1]
 [   0 1125    2    2    0    1    2    1    2    0]
 [   5    0  996   10    3    0    2    7    8    1]
 [   2    0    3  984    0    4    0    7    4    6]
 [   0    0    3    0  949    0    6    0    3   21]
 [   2    2    2    9    1  858    5    4    7    2]
 [   5    3    0    0    3    4  937    0    6    0]
 [   1    4   19    6    1    0    0  984    2   11]
 [   4    1    3    3    2    2    4    2  942   11]
 [   4    5    2    6    8    1    0    5    6  972]]
Classification report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       980
           1       0.99      0.99      0.99      1135
           2       0.97      0.97      0.97      1032
           3       0.96      0.97      0.97      1010
           4       0.98      0.97      0.97       982
           5       0.98      0.96      0.

In [12]:
# training XGBoost with PCA
from xgboost import XGBClassifier

xgb_clf_PCA = XGBClassifier(
    objective='multi:softmax',
    num_class=10,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)

start = time.perf_counter()
xgb_clf_PCA.fit(X_train_pca, y_train)
xgb_training_time = time.perf_counter() - start

y_pred_PCA = xgb_clf_PCA.predict(X_test_pca)
xgb_acc = accuracy_score(y_test, y_pred_PCA)
print(f"XGBoost accuracy: {xgb_acc:.4f}")
print(f"XGBoost training time: {xgb_training_time:.2f} seconds")

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_PCA))

print("Classification report:")
print(classification_report(y_test, y_pred_PCA))


XGBoost accuracy: 0.9526
XGBoost training time: 10.11 seconds
Confusion matrix:
[[ 962    0    3    1    0    2    8    2    2    0]
 [   0 1119    2    2    1    1    5    1    4    0]
 [   4    1  976   13    4    4    5   10   13    2]
 [   2    0    4  963    1   11    1   10   16    2]
 [   1    1    3    0  932    1   10    3    3   28]
 [   5    2    1   17    6  835    6    4   10    6]
 [  11    2    3    1    5    6  926    0    3    1]
 [   1    7   15    3    6    0    0  974    2   20]
 [   5    0    6   16    6   18    4    9  899   11]
 [   4    6    0    7   20   11    1    9   11  940]]
Classification report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.97       980
           1       0.98      0.99      0.98      1135
           2       0.96      0.95      0.95      1032
           3       0.94      0.95      0.95      1010
           4       0.95      0.95      0.95       982
           5       0.94      0.94      0.

### Questions to Consider:
- How much does PCA help when using SGDClassifier compared to Random Forest?

In comparison with Random Forestt and SGDClassifier, SGDClassifier did alot better using PCA than Random Forest,
SGDClassifier was able to improve in both performnce and training time with 87.4% baseline test accuracy to 89.59% PCA test accuracy and 124.15s baseline train time to 24.36s PCA train time

- Why might the effect differ between model types?

SGD is a linear model, so reducing noisy/redundant features with PCA can make optimization easier and faster. Random Forest already handles high-dimensional raw features well via tree splits and feature subsampling, while PCA can remove/blur useful nonlinear structure, so it may hurt accuracy

## Part 2 — Reflection & Summary
Write a short reflection (1–2 paragraphs) addressing:
1. What did you learn about PCA in practice?

- Consider effects on training time, model performance, and data dimensionality.

2. Any observations or surprises?

- Did PCA help some models more than others?

- How did dimensionality reduction affect accuracy

I learned that with applying PCA, you treat it like StandardScaler() to your training and test sets. It basically a good practice technique to help 'spcifc models' improve in both time and performnace. PCA should only be used in spcfic cases, 1, when you have large amount of features, with PCA you are able to reduce a large feature set to features that will help a model, basaiicaly get ride of noice. with PCA applied on this data, it went from 784 featurees to 154 being used. This just makes training in gereral more efficient.

One thing that i was surprised about was the amount of features taken out with PCA and how much time was saved with PCA. When applied correctly, PCA was able to reduce training time by around 100s while still improving test accuracy. Usally i would think that a model that needs to train longer, mean i will proabally have a high accuracy. But that was before taking this course. a models perfomance is really dependent on other things like the math behind it and other things. With the amount of feature reduced down by over 500 features, Im surprised that the accuracy improved rather than got close to the base line. 